# DEE Test T³ — Extensión: régimen de desorden y N grande

**Contexto:** Los tests anteriores mostraron que:
- Grilla perturbada (jitter 0.25): dispersión 2.81% ← sexteto T³ limpio
- Uniforme puro: dispersión 20.82% ← sexteto borroneado

**Preguntas abiertas que este notebook resuelve:**

1. **¿Cuánto desorden tolera el test antes de romperse?** Barremos jitter de 0 (grilla perfecta) a 1 (equivalente a uniforme) y vemos dónde se degrada la señal.

2. **¿El uniforme puro converge a cero con N, o se estanca?** Esto distingue "dispersión por ruido estadístico" de "dispersión por fallo estructural del kernel".

3. **¿Qué régimen de desorden corresponde al sustrato DEE físico?** Esto no lo resolvemos aquí pero queda claro el espacio de parámetros.

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
import time

PLOT_DIR = 'plots_T3_extendido'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name, fig=None, dpi=120):
    if fig is None:
        fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → {path}')

print(f'Plots se guardarán en: {PLOT_DIR}/')

## 1. Funciones base

In [ ]:
def periodic_distance_matrix(points, L=1.0):
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def build_DEE_laplacian(points, k_neighbors=30, alpha=2.0, L=1.0):
    N = len(points)
    D_mat = periodic_distance_matrix(points, L)
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    neighbor_idx = np.argpartition(D_search, k_neighbors, axis=1)[:, :k_neighbors]
    
    rows, cols, vals = [], [], []
    for i in range(N):
        for j in neighbor_idx[i]:
            d = D_mat[i, j]
            if d > 0:
                w = 1.0 / d**alpha
                rows.append(i)
                cols.append(j)
                vals.append(w)
    
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    degrees = np.array(W.sum(axis=1)).flatten()
    D_diag = diags(degrees)
    return D_diag - W

def get_eigenvalues(L_sparse, n_eig=25):
    try:
        eigs, _ = eigsh(L_sparse, k=n_eig, which='SM', tol=1e-8)
    except Exception:
        eigs_all = np.linalg.eigvalsh(L_sparse.toarray())
        eigs = eigs_all[:n_eig]
    return np.sort(eigs)

def dispersion_sexteto(eigs):
    s = eigs[1:7]
    return (s.max() - s.min()) / s.max()

def grid_perturbed_sample(N_target, jitter_fraction, seed=42):
    """
    Genera puntos en grilla regular con jitter controlable.
    
    jitter_fraction:
      0.0 = grilla perfecta (100% ordenada)
      0.5 = jitter medio (cada punto se mueve hasta la mitad del spacing)
      1.0 = jitter máximo físicamente razonable (se mueve hasta 1 spacing)
      
    Cuando jitter ≳ 1, los puntos pueden intercambiarse posiciones con
    vecinos, y la distribución se acerca a uniforme puro.
    """
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    
    # Jitter: cada punto se mueve en cada eje en [-jitter·spacing, +jitter·spacing]
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

print('Funciones base listas.')

## 2. Test D — Barrido de jitter

Barremos el parámetro de jitter entre 0 (grilla perfecta) y 1 (desorden máximo).

**Tamaño fijo:** N=1331 (=11³) para consistencia con el test anterior.
**k=30** (el que resultó mejor en el diagnóstico).
**Repeticiones:** 3 semillas por jitter para estimar dispersión.

**Lo que esperamos ver:**
- jitter=0: dispersión muy baja (~0%) pero el test es trivial
- jitter=0.25: ~3% (ya lo sabemos)
- jitter=0.5-1.0: degradación progresiva
- jitter≈1.0: debería acercarse al 20% del uniforme puro

In [ ]:
jitter_values = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5]
seeds = [42, 123, 2024]
N_target = 1331  # 11^3
k_test = 30

results_jitter = {}

print(f'Barrido de jitter (N={N_target}, k={k_test}, {len(seeds)} semillas por punto):')
print(f"{'jitter':>8} {'disp media':>12} {'disp ±std':>12} {'λ₇/λ₆ medio':>14}")
print('-'*55)

t0 = time.time()
for j in jitter_values:
    disps = []
    ratios = []
    for seed in seeds:
        points, N_actual = grid_perturbed_sample(N_target, j, seed=seed)
        L_mat = build_DEE_laplacian(points, k_neighbors=k_test, alpha=2.0, L=1.0)
        eigs = get_eigenvalues(L_mat, n_eig=20)
        disps.append(dispersion_sexteto(eigs))
        ratios.append(eigs[7]/eigs[6])
    
    disp_mean = np.mean(disps)
    disp_std = np.std(disps)
    ratio_mean = np.mean(ratios)
    
    results_jitter[j] = {
        'disp_mean': disp_mean,
        'disp_std': disp_std,
        'ratio_mean': ratio_mean,
        'disps': disps
    }
    
    print(f'{j:>8.2f} {disp_mean*100:>10.2f}%  ±{disp_std*100:>6.2f}%  {ratio_mean:>14.3f}')

print(f'\nTiempo total: {time.time()-t0:.1f}s')

In [ ]:
# Plot del barrido de jitter
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

js = list(results_jitter.keys())
means = [results_jitter[j]['disp_mean']*100 for j in js]
stds = [results_jitter[j]['disp_std']*100 for j in js]

# Panel 1: dispersión vs jitter
ax1.errorbar(js, means, yerr=stds, fmt='o-', markersize=10, linewidth=2,
             color='darkblue', capsize=5, label='DEE (α=2, k=30)')
ax1.axhline(10, linestyle='--', color='green', alpha=0.5, label='Umbral "bueno" (10%)')
ax1.axhline(20.82, linestyle=':', color='red', alpha=0.5,
            label='Referencia uniforme puro N=1200 (20.8%)')
ax1.set_xlabel('Jitter (fracción del spacing de grilla)', fontsize=12)
ax1.set_ylabel('Dispersión del sexteto [%]', fontsize=12)
ax1.set_title('Degradación de T³ con desorden creciente', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Panel 2: ratio λ₇/λ₆ vs jitter
ratios = [results_jitter[j]['ratio_mean'] for j in js]
ax2.plot(js, ratios, 'o-', markersize=10, linewidth=2, color='darkred')
ax2.axhline(2.0, linestyle='--', color='green', alpha=0.6, label='Teórico T³: 2.0')
ax2.set_xlabel('Jitter', fontsize=12)
ax2.set_ylabel('λ₇ / λ₆', fontsize=12)
ax2.set_title('Gap espectral entre sexteto y 12-plete', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
save_plot('06_barrido_jitter')
plt.show()

print('\nInterpretación:')
print('- Si la dispersión crece suavemente con jitter, la degradación es continua')
print('- Si hay una transición brusca en algún valor de jitter, hay umbral crítico')
print('- El "régimen de sustrato DEE físico" vive a la derecha del plot (desorden alto)')

## 3. Test E — Convergencia del uniforme puro con N grande

El test anterior mostró que en uniforme puro la dispersión decrece como N^(-1/2). Extrapolando, a N=5000 debería estar alrededor del 9%, y a N=10000 alrededor del 6%.

**Pregunta clave:** ¿se mantiene esa pendiente o se estanca en algún valor?

Si se mantiene, el uniforme puro **sí** converge al T³ correcto, solo que lentamente. Si se estanca en algún valor nonzero, hay un problema estructural.

**Tamaños:** 1200, 2000, 3000, 5000 (quizás 8000 si Colab aguanta).
**3 semillas por punto** para barras de error.

In [ ]:
N_values = [1200, 2000, 3000, 5000]
seeds_uni = [42, 123, 2024]

print('Uniforme puro (k=30, α=2):')
print(f"{'N':>6} {'disp media':>12} {'disp ±std':>12} {'tiempo':>10}")
print('-'*50)

results_N = {}
t_total = time.time()

for N in N_values:
    t0 = time.time()
    disps = []
    for seed in seeds_uni:
        rng = np.random.default_rng(seed)
        points = rng.uniform(0, 1, size=(N, 3))
        L_mat = build_DEE_laplacian(points, k_neighbors=30, alpha=2.0, L=1.0)
        eigs = get_eigenvalues(L_mat, n_eig=20)
        disps.append(dispersion_sexteto(eigs))
    
    disp_mean = np.mean(disps)
    disp_std = np.std(disps)
    elapsed = time.time() - t0
    
    results_N[N] = {'mean': disp_mean, 'std': disp_std, 'disps': disps}
    print(f'{N:>6} {disp_mean*100:>10.2f}%  ±{disp_std*100:>6.2f}%  {elapsed:>8.1f}s')

print(f'\nTiempo total test E: {time.time()-t_total:.1f}s')

In [ ]:
# Plot convergencia uniforme puro
fig, ax = plt.subplots(figsize=(10, 6))

Ns = sorted(results_N.keys())
means = np.array([results_N[n]['mean'] for n in Ns])
stds = np.array([results_N[n]['std'] for n in Ns])

ax.errorbar(Ns, means*100, yerr=stds*100, fmt='o-', markersize=12,
            linewidth=2, color='darkred', capsize=6,
            label='DEE uniforme puro (3 semillas)')

# Ajuste power law: log(disp) = a + b*log(N)
N_arr = np.array(Ns)
log_N = np.log(N_arr)
log_disp = np.log(means)
slope, intercept = np.polyfit(log_N, log_disp, 1)

# Extrapolación
N_extrap = np.logspace(np.log10(1000), np.log10(30000), 100)
disp_fit = np.exp(intercept) * N_extrap**slope
ax.plot(N_extrap, disp_fit*100, '--', color='gray', alpha=0.7,
        label=f'Ajuste: disp ∝ N^{{{slope:.2f}}}')

# Líneas de referencia
ax.plot(N_extrap, 30*(N_extrap/500)**(-0.5), ':', color='lightblue', alpha=0.8,
        label='N^(−1/2) ideal')
ax.axhline(10, linestyle='--', color='green', alpha=0.5, label='Umbral "bueno"')
ax.axhline(5, linestyle=':', color='darkgreen', alpha=0.5, label='Umbral "excelente"')

# Proyección a qué N llegaría al umbral bueno
N_to_good = (0.10 / np.exp(intercept))**(1/slope) if slope < 0 else np.inf
if N_to_good < 1e6:
    ax.axvline(N_to_good, color='orange', alpha=0.5, linestyle=':',
               label=f'N para alcanzar 10%: ~{int(N_to_good)}')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('N (nodos)', fontsize=12)
ax.set_ylabel('Dispersión del sexteto [%]', fontsize=12)
ax.set_title(f'Uniforme puro: ¿converge o se estanca? (slope ajustado: {slope:.2f})', fontsize=13)
ax.legend(fontsize=9, loc='best')
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
save_plot('07_convergencia_uniforme_grande')
plt.show()

print(f'\nAjuste: dispersión ∝ N^{slope:.3f}')
print(f'Si slope ≈ -0.5, convergencia ideal confirmada')
print(f'Si slope >> -0.5 (menos negativo), convergencia más lenta → posible problema')
if slope < 0 and N_to_good < 1e6:
    print(f'\nExtrapolación: para alcanzar dispersión 10% con uniforme puro se necesita N≈{int(N_to_good)}')
else:
    print('\nAtención: la extrapolación sugiere que el uniforme puro no converge al régimen bueno')

## 4. Test F — Plot combinado: qué régimen corresponde al sustrato DEE

Unificamos los resultados en un diagrama: dispersión en el plano (jitter, N).

El "sustrato DEE físico" debería caer en alguna región de ese plano; saber dónde es informativo.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

# Curva jitter (a N fijo=1331)
js = list(results_jitter.keys())
disp_j = [results_jitter[j]['disp_mean']*100 for j in js]
ax.plot(js, disp_j, 'o-', markersize=11, linewidth=2.5,
        color='darkblue', label='Barrido de jitter (N=1331, k=30)')

# Punto del uniforme puro (jitter=1 equivalente) — lo ubicamos a jitter=1.0
# para comparación visual
ax.plot([1.0], [results_N[min(N_values)]['mean']*100], 's',
        markersize=14, color='darkred', label=f'Uniforme puro N={min(N_values)}')
ax.plot([1.0], [results_N[max(N_values)]['mean']*100], '^',
        markersize=14, color='orange', label=f'Uniforme puro N={max(N_values)}')

# Flecha conectando
ax.annotate('', xy=(1.0, results_N[max(N_values)]['mean']*100),
            xytext=(1.0, results_N[min(N_values)]['mean']*100),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.text(1.02, (results_N[min(N_values)]['mean']+results_N[max(N_values)]['mean'])*50,
        f'N: {min(N_values)} → {max(N_values)}', fontsize=10, color='gray')

# Regiones cualitativas
ax.axhline(5, linestyle=':', color='darkgreen', alpha=0.5)
ax.axhline(10, linestyle='--', color='green', alpha=0.5)
ax.axhspan(0, 5, alpha=0.08, color='green', label='Excelente (<5%)')
ax.axhspan(5, 10, alpha=0.05, color='limegreen', label='Bueno (5-10%)')
ax.axhspan(10, 25, alpha=0.04, color='orange', label='Problemático (>10%)')

ax.set_xlabel('Jitter (orden → desorden)', fontsize=12)
ax.set_ylabel('Dispersión del sexteto [%]', fontsize=12)
ax.set_title('Régimen de emergencia de T³ en DEE', fontsize=13)
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)
ax.set_ylim(-1, 28)
plt.tight_layout()
save_plot('08_diagrama_regimenes')
plt.show()

## 5. Síntesis y empaquetado

In [ ]:
print('='*70)
print('SÍNTESIS — Test T³ extendido')
print('='*70)
print()
print('Test D — Barrido de jitter (N=1331, k=30):')
for j in sorted(results_jitter.keys()):
    r = results_jitter[j]
    marker = ''
    if r['disp_mean']*100 < 5: marker = ' ← excelente'
    elif r['disp_mean']*100 < 10: marker = ' ← bueno'
    print(f'  jitter={j:4.2f}: {r["disp_mean"]*100:5.2f}% ± {r["disp_std"]*100:4.2f}%  '
          f'λ₇/λ₆={r["ratio_mean"]:.2f}{marker}')

print()
print(f'Test E — Convergencia uniforme puro (k=30, α=2):')
for N in sorted(results_N.keys()):
    r = results_N[N]
    print(f'  N={N:5d}: {r["mean"]*100:5.2f}% ± {r["std"]*100:4.2f}%')
print(f'  Slope ajustado: {slope:.3f}  (ideal: -0.5)')

print()
print('='*70)
print('CONCLUSIÓN')
print('='*70)

# Transición de jitter
j_critical = None
for j in sorted(results_jitter.keys()):
    if results_jitter[j]['disp_mean']*100 > 10:
        j_critical = j
        break

if j_critical is not None:
    print(f'\n• El sexteto T³ emerge limpiamente (<10%) para jitter < {j_critical:.2f}')
    print(f'  Se degrada para jitter ≥ {j_critical:.2f}')
else:
    print(f'\n• El sexteto T³ emerge limpiamente en todo el rango de jitter probado')

# Convergencia N
if abs(slope - (-0.5)) < 0.15:
    print(f'\n• El uniforme puro converge como N^(-1/2) (slope={slope:.2f})')
    print('  → es convergencia ideal pero lenta; N muy grande requerido')
elif slope > -0.3:
    print(f'\n• El uniforme puro converge más lento que N^(-1/2) (slope={slope:.2f})')
    print('  → posible problema estructural del kernel en muestreos desordenados')
else:
    print(f'\n• Convergencia anómala (slope={slope:.2f})')

print(f'\n• Para alcanzar 10% dispersión con uniforme puro se necesita N≈{int(N_to_good) if N_to_good<1e6 else "∞"}')

print('\n' + '='*70)
print('IMPLICACIONES PARA EL DOCUMENTO DEE')
print('='*70)
print('''
Este test muestra con evidencia cuantitativa que el sexteto T³ del §3.7
es sensible al tipo de muestreo. El documento debería especificar en qué
régimen de desorden vive el sustrato físico DEE, y si ese régimen es
compatible con la emergencia limpia de geometría T³.

Dos lecturas posibles:
  (a) El sustrato DEE naturalmente cristaliza → el jitter bajo es el
      régimen físico correcto, y T³ emerge limpiamente. Pero esto hay
      que demostrarlo dinámicamente, no asumirlo.
  (b) El sustrato DEE es desordenado → T³ emerge solo a N muy grande,
      potencialmente incompatible con simulaciones actuales.

Este resultado debería incluirse en la próxima versión del documento.
''')

In [ ]:
# Empaquetar plots
import shutil
shutil.make_archive('plots_T3_extendido', 'zip', PLOT_DIR)
print(f'\nZip creado: plots_T3_extendido.zip ({len(os.listdir(PLOT_DIR))} plots)')

try:
    from google.colab import files
    files.download('plots_T3_extendido.zip')
    print('→ Descarga iniciada')
except ImportError:
    pass